In [3]:
import json
import os
from pathlib import Path
from statistics import mean

# =============================================================================
# Configuration
# =============================================================================
FOLDER_PATH = "./raw_tree"          # folder containing the JSON files
OUTPUT_FILE = "evaluation_report.txt"

# List of statistics fields to average (must exist in metadata.statistics)
STATS_TO_AVERAGE = [
    "total_nodes",
    "api_calls",
    "solutions_found",
    "code_executions",
    "code_errors",
    "deadend_memory_skipped",
    "total_input_tokens",
    "total_output_tokens",
    "api_saved_two_num_cache"
]

# =============================================================================
# Process a single JSON file
# =============================================================================
def process_file(filepath: Path):
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Extract statistics
    metadata = data.get("metadata", {})
    stats = metadata.get("statistics", {})

    # Get solutions_found count (default 0 if missing)
    solutions_found = stats.get("solutions_found", 0)

    # Success if solutions_found > 0
    success = solutions_found > 0

    return {
        "file": filepath.name,
        "stats": stats,
        "solutions_found": solutions_found,
        "success": success
    }

# =============================================================================
# Main aggregation and reporting
# =============================================================================
def main():
    folder = Path(FOLDER_PATH)
    if not folder.exists():
        print(f"Folder '{FOLDER_PATH}' does not exist.")
        return

    json_files = list(folder.glob("*.json"))
    if not json_files:
        print(f"No JSON files found in '{FOLDER_PATH}'.")
        return

    results = []
    for f in json_files:
        try:
            res = process_file(f)
            results.append(res)
        except Exception as e:
            print(f"Error processing {f.name}: {e}")

    total_files = len(results)
    successful_files = sum(1 for r in results if r["success"])
    success_rate = successful_files / total_files if total_files > 0 else 0.0

    # Compute averages for requested statistics
    averages = {}
    for key in STATS_TO_AVERAGE:
        values = [r["stats"].get(key, 0) for r in results]
        averages[key] = mean(values) if values else 0.0

    # Write report
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as out:
        out.write("=" * 70 + "\n")
        out.write("GAME 24 TREE SEARCH EVALUATION REPORT\n")
        out.write("=" * 70 + "\n\n")

        out.write(f"Total JSON files processed: {total_files}\n")
        out.write(f"Files with solutions_found > 0: {successful_files}\n")
        out.write(f"Success rate: {success_rate:.2%}\n\n")

        out.write("-" * 50 + "\n")
        out.write("AVERAGE STATISTICS (across all files)\n")
        out.write("-" * 50 + "\n")
        for key, val in averages.items():
            out.write(f"{key:25s}: {val:10.2f}\n")

        out.write("\n" + "-" * 50 + "\n")
        out.write("DETAILED FILE RESULTS\n")
        out.write("-" * 50 + "\n")
        for r in results:
            status = "SOLUTION FOUND" if r["success"] else "NO SOLUTION"
            out.write(f"{r['file']:50s} solutions_found={r['solutions_found']:<2} {status}\n")

    print(f"Report written to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

Report written to evaluation_report.txt
